# 41 Computational cost
Weights and inference time of the ten Paper 2 models on one NVIDIA T4. Random 256x256 inputs, no data and no training. Batch 1 latency and per image time at batch 16, each the median of 300 or 60 timed calls after warm up, repeated three times.

In [ ]:
# Paper 2: computational cost (no data needed; random inputs, no training). Runtime: T4 GPU.
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, time, json
import numpy as np, pandas as pd, tensorflow as tf
gpus = tf.config.list_physical_devices('GPU'); assert gpus, 'No GPU: Runtime > Change runtime type > T4 GPU'
from tensorflow.python.client import device_lib
name = [d.physical_device_desc for d in device_lib.list_local_devices() if d.device_type == 'GPU'][0]
print(name); assert 'T4' in name, 'This must run on a T4 (the training GPU). Reconnect until a T4 is assigned.'
PROJECT = '/content/drive/MyDrive/quantum-placement'
CODE_DIR = f'{PROJECT}/code'
if not os.path.exists(f'{CODE_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True); assert hits, 'qseg package not found on Drive'
    CODE_DIR = os.path.dirname(os.path.dirname(hits[0]))
for p in (CODE_DIR, f'{PROJECT}/code'):
    if p not in sys.path: sys.path.insert(0, p)
import qseg2, qseg2_rev4; qseg2_rev4.patch()
OUT = f'{PROJECT}/results/compute_cost'; os.makedirs(OUT, exist_ok=True)
print('TF', tf.__version__)

In [ ]:
MODELS = ['full', 'unet', 'g_q', 'g_cl', 'g_tr', 'g_id', 'p_q', 'p_cl', 'p_tr', 'p_id']
BLOCK = {'q': 'quantum_circuit', 'cl': 'classical_replacement', 'tr': 'trig_replacement'}
rng = np.random.default_rng(0)
x1 = tf.constant(rng.random((1, 256, 256, 1), dtype=np.float32))
x16 = tf.constant(rng.random((16, 256, 256, 1), dtype=np.float32))

def timed(f, x, warm, reps):
    for _ in range(warm): f(x).numpy()
    t = []
    for _ in range(reps):
        s = time.perf_counter(); f(x).numpy(); t.append(time.perf_counter() - s)
    return np.array(t) * 1000.0

rows = []
for m in MODELS:
    tf.keras.backend.clear_session(); tf.random.set_seed(0)
    model = qseg2.build_p2_model(m)
    f = tf.function(lambda x: model(x, training=False))
    kind = m.split('_')[1] if '_' in m else None
    blk = 0
    if kind in BLOCK:
        blk = int(sum(np.prod(w.shape) for w in model.get_layer(BLOCK[kind]).trainable_weights))
    r = dict(model=m, block_weights=blk,
             trainable=int(sum(np.prod(w.shape) for w in model.trainable_weights)),
             total=int(model.count_params()))
    for rep in range(3):                      # three repeats; the median of the three medians is reported
        a = timed(f, x1, 30, 300); b = timed(f, x16, 10, 60) / 16
        r[f'ms_b1_rep{rep}'] = float(np.median(a)); r[f'ms_img_b16_rep{rep}'] = float(np.median(b))
    r['ms_batch1'] = float(np.median([r[f'ms_b1_rep{i}'] for i in range(3)]))
    r['ms_per_image_batch16'] = float(np.median([r[f'ms_img_b16_rep{i}'] for i in range(3)]))
    r['gpu'] = name; r['tf'] = tf.__version__
    rows.append(r); print(m, r['block_weights'], r['trainable'], round(r['ms_batch1'], 2), round(r['ms_per_image_batch16'], 3))
df = pd.DataFrame(rows); df.to_csv(f'{OUT}/inference_ms.csv', index=False)
print(df[['model', 'block_weights', 'trainable', 'total', 'ms_batch1', 'ms_per_image_batch16']].round(3).to_string(index=False))
print('COMPUTE COST OK ->', f'{OUT}/inference_ms.csv')

In [ ]:
from google.colab import runtime
drive.flush_and_unmount(); print('Drive flushed'); time.sleep(20); runtime.unassign()